# Bài học 13 - Bộ nhớ của đại lý


## Thiết lập

Notebook này trình bày cách xây dựng một đại lý đặt chuyến du lịch với **bộ nhớ bền vững** sử dụng **Microsoft Agent Framework** (MAF).

Bạn sẽ học cách các loại bộ nhớ đại lý khác nhau — bộ nhớ làm việc, bộ nhớ ngắn hạn và bộ nhớ dài hạn — ảnh hưởng đến cách đại lý lưu giữ và sử dụng thông tin trong các cuộc trò chuyện.

**Yêu cầu trước:**
- Một dự án Azure AI Foundry với mô hình chat đã triển khai (ví dụ `gpt-4o-mini`).
- Đăng nhập bằng Azure CLI — chạy `az login` trong terminal của bạn.
- `AZURE_AI_PROJECT_ENDPOINT` — điểm cuối dự án Azure AI Foundry của bạn.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — tên mô hình bạn đã triển khai.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Các loại bộ nhớ của tác nhân

Các tác nhân AI có thể sử dụng các loại bộ nhớ khác nhau, mỗi loại phục vụ một mục đích riêng biệt:

### Bộ nhớ làm việc
Chuỗi hội thoại chính — các tin nhắn được trao đổi trong một phiên làm việc duy nhất. Tác nhân có thể tham khảo lại các tin nhắn trước đó trong cùng một chuỗi để duy trì sự nhất quán. Trong MAF, điều này được tạo ra thông qua **`agent.create_session()`**, trả về một `AgentSession`.

### Bộ nhớ ngắn hạn
Thông tin tồn tại trong suốt thời gian của một nhiệm vụ hoặc phiên làm việc nhưng không được lưu trữ vĩnh viễn. Ví dụ, tác nhân có thể tích lũy các sự thật trong quá trình hội thoại lập kế hoạch nhiều lượt và sử dụng chúng để tạo ra một lịch trình cuối cùng.

### Bộ nhớ dài hạn
Sở thích và các sự thật tồn tại **qua các phiên làm việc**. Người dùng trở lại sẽ không phải lặp lại các hạn chế về chế độ ăn uống hoặc phong cách du lịch của họ. Bộ nhớ dài hạn thường được hỗ trợ bởi một kho ngoại vi — cơ sở dữ liệu, tệp, hoặc chỉ mục vector — và được hiển thị cho tác nhân thông qua các công cụ.


## Bộ nhớ làm việc với phiên

Hình thức bộ nhớ đơn giản nhất là phiên hội thoại. Khi bạn truyền cùng một phiên (được tạo qua `agent.create_session()`) vào các lần gọi `agent.run()` liên tiếp, đại lý sẽ thấy toàn bộ lịch sử của cuộc trò chuyện đó và có thể nhớ lại các chi tiết trước đó.

Hãy tạo một đại lý du lịch và trình bày bộ nhớ làm việc.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Đại lý đã nhớ đúng ngân sách vì cả hai tin nhắn đều chia sẻ cùng một phiên làm việc. Đây là **bộ nhớ làm việc** — nó chỉ tồn tại trong suốt thời gian của phiên làm việc.

### Điều gì xảy ra với một chuỗi mới?

Nếu chúng ta tạo một phiên làm việc **mới**, đại lý sẽ không có ký ức về cuộc trò chuyện trước đó:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Mẫu Bộ Nhớ Dài Hạn

Để ghi nhớ sở thích của người dùng **trong nhiều phiên làm việc**, chúng ta cần một kho lưu trữ bền vững tồn tại bên ngoài chuỗi hội thoại. Đại lý truy cập kho lưu trữ này thông qua **công cụ** — các hàm mà nó có thể gọi để lưu và truy xuất thông tin.

Dưới đây chúng ta triển khai một kho lưu trữ sở thích đơn giản trong bộ nhớ (trong thực tế, bạn sẽ sử dụng cơ sở dữ liệu hoặc chỉ mục vector làm nền tảng) và cung cấp nó dưới dạng các công cụ mà đại lý có thể sử dụng.

### Kiến trúc
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Kịch bản 1 — Người dùng lần đầu đặt chuyến đi kỷ niệm

Sarah ghé thăm lần đầu tiên. Đại lý nên lưu sở thích của cô ấy qua các công cụ và sử dụng chúng để đề xuất các khách sạn.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Tình huống 2 — Sarah trở lại sau vài tuần

Sarah bắt đầu một **chủ đề mới hoàn toàn** (mô phỏng một phiên làm việc mới). Bộ nhớ làm việc trống, nhưng kho lưu trữ sở thích dài hạn vẫn còn thông tin của cô ấy. Đại lý nên lấy lại nó và sử dụng để cá nhân hóa các đề xuất.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Tóm tắt

Trong bài học này, bạn đã học về ba loại bộ nhớ của tác nhân và cách triển khai chúng với Microsoft Agent Framework:

| Loại Bộ nhớ | Cơ chế MAF | Thời gian tồn tại |
|---|---|---|
| **Làm việc** | `agent.create_session()` | Một cuộc trò chuyện đơn |
| **Ngắn hạn** | Ngữ cảnh tích lũy trong một luồng | Một nhiệm vụ / phiên làm việc |
| **Dài hạn** | Kho bên ngoài truy cập qua các hàm `@tool` | Qua nhiều phiên làm việc |

### Những điểm chính cần nhớ
1. **`agent.create_session()`** cung cấp bộ nhớ làm việc — tác nhân thấy toàn bộ lịch sử cuộc trò chuyện trong một phiên.
2. **Phiên mới làm mất ngữ cảnh** — nếu không có bộ nhớ dài hạn, tác nhân không thể nhớ các cuộc trò chuyện trước đó.
3. **Các hàm `@tool`** tạo cầu nối — cho phép tác nhân lưu trữ và truy xuất thông tin từ kho lưu trữ bền vững.
4. **Cá nhân hóa cải thiện theo thời gian** — càng lưu nhiều sở thích, các đề xuất của tác nhân càng chính xác.

### Ứng dụng thực tế
- **Dịch vụ khách hàng**: Ghi nhớ lịch sử và sở thích của khách hàng
- **Trợ lý cá nhân**: Duy trì ngữ cảnh qua nhiều ngày hoặc tuần
- **Chăm sóc sức khỏe**: Theo dõi thông tin và sở thích của bệnh nhân
- **Thương mại điện tử**: Mua sắm cá nhân hóa dựa trên lịch sử

### Bước tiếp theo
- Thay thế dict trong bộ nhớ bằng cơ sở dữ liệu hoặc kho vector (ví dụ: Azure AI Search)
- Thêm thời hạn hết hạn bộ nhớ cho thông tin nhạy cảm theo thời gian
- Xây dựng hệ thống đa tác nhân với bộ nhớ chia sẻ
- Khám phá sổ tay Cognee cho bộ nhớ dựa trên kiến thức đồ thị


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:  
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi nỗ lực đảm bảo độ chính xác, xin lưu ý rằng các bản dịch tự động có thể chứa lỗi hoặc không chính xác. Tài liệu gốc bằng ngôn ngữ gốc của nó nên được xem là nguồn tham khảo chính thức. Đối với thông tin quan trọng, khuyến nghị sử dụng dịch vụ dịch thuật chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ sự hiểu nhầm hoặc diễn giải sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
